In [15]:
import pandas as pd
import os
from sqlalchemy import create_engine
import numpy as np
import re

def remove_links_from_excel(excel_path, output_path):
    from openpyxl import load_workbook

    # Load the workbook
    workbook = load_workbook(excel_path)
    sheet = workbook.active

    # Iterate through cells in the sheet
    for row in sheet.iter_rows():
        for cell in row:
            if cell.hyperlink:
                # Store the cell value
                cell_value = cell.value
                # Remove the hyperlink
                cell.hyperlink = None
                # Restore the cell value
                cell.value = cell_value

    # Save the modified workbook
    workbook.save(output_path)

def create_db_from_excel1(excel_path, db_path):
    """
    Reads an Excel file and creates a SQLite database from its sheets.
    Each sheet is converted into a separate table with proper header handling.
    """
    if os.path.exists(db_path):
        os.remove(db_path)  # Remove existing DB to start fresh
        
    xls = pd.ExcelFile(excel_path)
    engine = create_engine(f"sqlite:///{db_path}")
    
    # Process each sheet in the Excel file
    for sheet_idx, sheet_name in enumerate(xls.sheet_names):
        try:
            # First, read the Excel sheet without setting headers
            if sheet_idx == 0:  # First table (summary table)
                # For the first table, use the first row as header
                df = pd.read_excel(xls, sheet_name=sheet_name)
                print(f"Processing summary table '{sheet_name}' with standard headers")

            elif sheet_idx == 2:
                # remove the top two rows and set the third row as header
                df = pd.read_excel(xls, sheet_name=sheet_name, header=2)

                # convert the first two columns to string
                df.iloc[:, 0:2] = df.iloc[:, 0:2].astype(str)

                hyperlink_pattern = r'=HYPERLINK\s*\(\s*"[^"]*"\s*,\s*"([^"]*)"\s*\)'
            
                # Get the first two column names
                first_two_cols = list(df.columns)[:2] if len(df.columns) >= 2 else list(df.columns)
                
                for col in first_two_cols:
                    # Apply the extraction to each cell in the column
                    df[col] = df[col].apply(lambda x: 
                                        re.search(hyperlink_pattern, str(x)).group(1) 
                                        if isinstance(x, str) and re.search(hyperlink_pattern, str(x)) 
                                        else x)
                print(f"Processing table '{sheet_name}' with standard headers")
                # display(df.head())  # Display the first few rows of the DataFrame

                # parse the first two columns as strings, split by comma

            elif sheet_idx == 5:  # Second table (summary table)
                # For the second table, use the first row as header
                df = pd.read_excel(xls, sheet_name=sheet_name)

                print(f"Processing summary table '{sheet_name}' with standard headers")

            elif sheet_idx == 8:  # Second table (detailed table)
                # For other tables, we need to handle the multi-row headers
                # Read the sheet with header=None to get all rows
                df = pd.read_excel(xls, sheet_name=sheet_name, header=None)
                
                if len(df) >= 3:  # Make sure we have enough rows
                    # Get the header rows
                    top_row = df.iloc[0].replace({np.nan: None, 'NaN': None})  # Row with potential suffixes
                    third_row = df.iloc[2].replace({np.nan: None, 'NaN': None})  # Row with base column names
                    
                    # Create combined headers
                    combined_headers = []
                    current_suffix = None
                    
                    for i in range(len(top_row)):
                        # Update suffix if we encounter a non-None value in the top row
                        if top_row[i] is not None:
                            current_suffix = str(top_row[i])
                        
                        # Get base column name from third row
                        base_name = str(third_row[i]) if third_row[i] is not None else f"col_{i}"
                        
                        # Create combined column name
                        if current_suffix:
                            combined_headers.append(f"{base_name}_{current_suffix}")
                        else:
                            combined_headers.append(base_name)
                    
                    # Set combined headers and drop the first three rows
                    df.columns = combined_headers
                    df = df.iloc[3:]
                    print(f"Processed table '{sheet_name}' with combined headers")
                    
                else:
                    print(f"Sheet '{sheet_name}' doesn't have enough rows for header processing")
            else:  # For other tables, use the first row as header
                continue
            
            # Reset index to ensure proper SQLite import
            df = df.reset_index(drop=True)
            
            # Clean column names for SQL compatibility
            df.columns = [str(col).replace(' ', '_').replace('(', '').replace(')', '').replace('.', '_')
                         .replace('-', '_').replace('/', '_').replace('\\', '_') for col in df.columns]
            
            # Sanitize table name
            table_name = ''.join(e for e in sheet_name if e.isalnum() or e == '_')
            if not table_name:  # if sheet name was all special chars
                table_name = f"table_{sheet_idx}"

            display(df)  # Display the first few rows of the DataFrame

            df.to_sql(table_name, engine, index=False, if_exists='replace')
            print(f"Sheet '{sheet_name}' imported as table '{table_name}' with {len(df)} rows.")
            
        except Exception as e:
            print(f"Could not import sheet '{sheet_name}': {e}")
    
    return engine

In [8]:
import pandas as pd
import os
from sqlalchemy import create_engine
from openpyxl import load_workbook


def create_db_from_excel(path, db_path):
    if os.path.exists(db_path):
        os.remove(db_path)

    engine = create_engine(f"sqlite:///{db_path}")

    # Load workbook and worksheet
    wb = load_workbook(path, data_only=True)
    ws = wb["By Variants"]

    # Read all rows into raw format
    all_data = [[cell.value for cell in row] for row in ws.iter_rows()]

    # Filter out rows that are completely empty
    all_data = [row for row in all_data if any(cell not in (None, "", "NULL", "–") for cell in row)]

    # Identify header row by matching keywords
    expected_keywords = ["Parent Item ID", "Parent Product Name", "Variant Count"]
    header_row_index = None
    for i, row in enumerate(all_data):
        row_strs = [str(cell).lower().strip() if cell else "" for cell in row]
        matches = sum(any(expected.lower() in cell for cell in row_strs) for expected in expected_keywords)
        if matches >= 2:
            header_row_index = i
            break

    if header_row_index is None:
        raise ValueError("Header row not found. Please check for expected column names.")

    # Extract header and data
    header = all_data[header_row_index]
    data = all_data[header_row_index + 1:]

    # Construct DataFrame and clean
    df = pd.DataFrame(data, columns=header)
    df = df.reset_index(drop=True)

    # Save to database
    df.to_sql("By_Variants", engine, index=False, if_exists='replace')
    print("✅ Cleaned data written to 'By_Variants' table.")



✅ Summary sheet saved.
✅ Detected true header at row index: 1
✅ 'By Variants' data written to database successfully.


In [16]:
import pandas as pd
import os
from sqlalchemy import create_engine
from openpyxl import load_workbook


def create_db_from_excel_display_only(path, db_path):
    if os.path.exists(db_path):
        os.remove(db_path)

    engine = create_engine(f"sqlite:///{db_path}")

    # === Step 1: Detect Header Row ===
    wb = load_workbook(filename=path, data_only=True)
    ws = wb['By Variants']

    expected_keywords = ["Parent Item ID", "Parent Product Name", "Variant Count"]
    header_row_index = None

    for i, row in enumerate(ws.iter_rows(values_only=True), start=1):
        row_values = [str(cell).strip().lower() if cell is not None else "" for cell in row]
        match_count = sum(any(expected.lower() in cell for cell in row_values) for expected in expected_keywords)
        if match_count >= 2:
            header_row_index = i
            break

    if header_row_index is None:
        raise ValueError("❌ Header row not found.")

    print(f"✅ Header row detected at Excel row: {header_row_index}")

    # === Step 2: Load DataFrame with Correct Header ===
    df = pd.read_excel(path, sheet_name="By Variants", header=header_row_index - 1, engine="openpyxl")

    # === Step 3: Replace the first column with clean display values ===
    product_ids = []
    for row in ws.iter_rows(min_row=header_row_index + 1, min_col=1, max_col=1):
        product_ids.append(row[0].value)
    df.iloc[:, 0] = product_ids

    # === Step 4: Clean Column Names ===
    df.columns = [
        str(col).strip()
        .replace(" ", "_")
        .replace("%", "pct")
        .replace("/", "_")
        .replace("-", "_")
        .replace("__", "_")
        for col in df.columns
    ]

    # === Step 5: Save to SQLite ===
    df.to_sql("By_Variants", engine, index=False, if_exists='replace')
    print("✅ Cleaned data saved to SQLite as 'By_Variants'")


# === USAGE ===
input_path = "Test1.xlsx"
output_db = "walmart_tacos.db"
create_db_from_excel_display_only(input_path, output_db)


✅ Header row detected at Excel row: 2


/var/folders/d4/lj0829c92ll3rl6b1qgvdvjw0000gn/T/ipykernel_35005/90515486.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, No

✅ Cleaned data saved to SQLite as 'By_Variants'


In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [23]:
import pandas as pd
import os
from sqlalchemy import create_engine
import numpy as np
import re


def create_db_from_excel(path, db_path):
    if os.path.exists(db_path):
        os.remove(db_path)  # Remove existing DB to start fresh
    
    xls = pd.ExcelFile(path)
    
    #Get the name of all the sheets in the excel file
    sheet_names = xls.sheet_names
    engine = create_engine(f"sqlite:///{db_path}")
    if sheet_names[0] == 'Summary':
        df = pd.read_excel(xls, sheet_name=sheet_names[0])
        table_name = 'Summary'
        # Clean column names for SQL compatibility
        df.to_sql(table_name, engine, index=False, if_exists='replace')
    if 'By Variants' in sheet_names:
        df_raw = pd.read_excel(xls, sheet_name='By Variants')
        required_headers = {"Parent Item ID", "Parent Product Name", "Variant Count"}

        # Find the first row that contains ALL required keywords
        for i, row in df_raw.iterrows():
            row_values = set(row.dropna().astype(str).str.strip())
            if required_headers.issubset(row_values):
                header_row_index = i
                break

        print(f"Detected true header at row index: {header_row_index}")

        # Load using detected header
        df_clean = pd.read_excel(path, sheet_name="By Variants", header=header_row_index)

        # Final cleanup
        df_clean = df_clean.reset_index(drop=True).dropna(axis=1, how='all')

        print(df_clean)

        # Clean column names for SQL compatibility
        # df.to_sql(table_name, engine, index=False, if_exists='replace')


    # Read the Excel file



In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [24]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected true header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             U

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [ ]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname

In [13]:
path = 'Test1.xlsx'
db_path = 'walmart_tacos.db'
engine = create_db_from_excel(path, db_path)

Detected header at row index: 0
      01/20/2025 - 02/18/2025 (30 days)           Unnamed: 1     Unnamed: 2  \
0                        Parent Item ID  Parent Product Name  Variant Count   
1                                   NaN                  NaN              -   
2                                   NaN                  NaN              -   
3                                   NaN                  NaN              1   
4                                   NaN                  NaN              -   
...                                 ...                  ...            ...   
23610                               NaN                  NaN              -   
23611                               NaN                  NaN              -   
23612                               NaN                  NaN              -   
23613                               NaN                  NaN              -   
23614                               NaN                  NaN              -   

             Unname